# Analytical Baselines

Hierarchical Roofline for F-DATA; PM100 calibrated resource-utilization power model, including the training-period-only coefficient calibration step (Decision #15).

See the conceptualization plan and EXPERIMENT_TRACKER.md for full context.

In [ ]:
import sys
sys.path.append("..")

import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config, features, metrics, plotting, models, roofline, splits, baselines

## Load data

This notebook doesn't touch F-DATA's `embedding` column at all, so the ~100GB memory constraint behind notebooks 01/02's random-file-then-sample pattern (Decision #7) doesn't apply here — `load_fdata_no_embedding` over a handful of months costs a few GB at most. To make the chronological split (Shared Context) meaningful — a real multi-month time range rather than a handful of scattered months — this notebook loads a **contiguous** run of F-DATA months instead, while still stopping short of all 38 (Decision #10's dev-scale default; full-scale F-DATA is deferred to notebook 05 onward). PM100 loads in full, as it has throughout — it's small enough not to need sampling.

In [ ]:
FDATA_DIR = "../data/raw/fdata"
PM100_PATH = "../data/raw/pm100/pm100_job_table.parquet"
N_CONSECUTIVE_MONTHS = 6  # dev-scale; contiguous so the chronological split spans real time

fdata_files = sorted(glob.glob(f"{FDATA_DIR}/*.parquet"))[:N_CONSECUTIVE_MONTHS]
print(f"Loading {len(fdata_files)} consecutive F-DATA months:",
      [f.split('/')[-1] for f in fdata_files])

fdata = features.load_fdata_no_embedding(fdata_files)
pm100 = pd.read_parquet(PM100_PATH)

print(f"\nF-DATA: {len(fdata):,} rows | PM100: {len(pm100):,} rows")

## Exclude non-completed jobs (Decision #4), reduce PM100's power trace to a scalar

Same filter as notebook 02 — a job that never finished doesn't have a meaningful duration or power draw to predict. PM100's `node_power_consumption` is a 20-second-interval time series per job (its native form, used later for LSTM/TCN), so for this notebook's per-job calibration/prediction task it's reduced to its per-job mean here, same as notebook 01's plotting code did.

In [ ]:
fdata = features.filter_completed_jobs(fdata, "fdata")
pm100 = features.filter_completed_jobs(pm100, "pm100")

pm100["power_mean"] = pm100["node_power_consumption"].apply(np.mean)

## Chronological split (Shared Context)

`src/splits.py` is the single source of truth for the split boundary — train on earlier submission times, test on later ones, 70/30 by row count — so every notebook from here on (this one, classical ML, DL, the hybrid model, evaluation) compares models against the same boundary rather than each notebook picking its own.

In [ ]:
fdata_train, fdata_test = splits.chronological_split(fdata, "fdata", train_frac=0.7)
pm100_train, pm100_test = splits.chronological_split(pm100, "pm100", train_frac=0.7)

print("F-DATA split boundary:", splits.split_boundary(fdata, "fdata"))
print(f"  train: {len(fdata_train):,} rows | test: {len(fdata_test):,} rows")
print()
print("PM100 split boundary:", splits.split_boundary(pm100, "pm100"))
print(f"  train: {len(pm100_train):,} rows | test: {len(pm100_test):,} rows")

# Part 1: F-DATA — Hierarchical Roofline (Decision #15)

F-DATA is the only one of the two datasets with real FLOP/performance-counter fields, so it's the only one that supports a genuine Roofline analysis. Three fields matter here:

- `flops` — total floating-point operations measured for the job (hardware performance counters).
- `mbwidth` — total bytes moved between memory and the compute cores over the job's lifetime.
- `opint` — operational/arithmetic intensity, precomputed by F-DATA's own authors as `flops / mbwidth` (checked below, not assumed).
- `pclass` — F-DATA's own precomputed compute-bound/memory-bound label for the job.

The ceiling comes from A64FX's published peak specs (Fugaku's node architecture): 3.3792 TFLOP/s double-precision peak and 1024 GB/s peak HBM2 bandwidth, both **per node**. Just like `avgpcon` (notebook 02's units investigation), `flops` and `mbwidth` turn out to be **job-wide totals across all allocated nodes**, not per-node quantities — checked directly below before the ceiling is applied, rather than assumed from the field names.

In [ ]:
rel_error = (np.abs(fdata["opint"] - fdata["flops"] / fdata["mbwidth"]) / fdata["opint"]).max()
print(f"opint vs. flops/mbwidth, max relative error across {len(fdata):,} jobs: {rel_error:.2e}")

perf = fdata["flops"] / fdata["duration"].replace(0, np.nan)
peak_single_node = roofline.A64FX_PEAK_DP_FLOPS_PER_NODE

n_exceed_raw = (perf > peak_single_node).sum()
n_exceed_per_node = (perf / fdata["nnuma"] > peak_single_node).sum()
print(f"\nJobs exceeding single-node peak FLOP/s, nnuma ignored:   {n_exceed_raw:,} / {len(fdata):,}")
print(f"Jobs exceeding single-node peak FLOP/s, divided by nnuma: {n_exceed_per_node:,} / {len(fdata):,}")
if n_exceed_raw:
    print("Median nnuma among the jobs that exceed it (nnuma ignored):",
          fdata.loc[perf > peak_single_node, "nnuma"].median())

In [ ]:
fdata_roofline = roofline.hierarchical_roofline_fdata(fdata)

agreement = (fdata_roofline["pclass"] == fdata_roofline["roofline_pclass"]).mean()
print(f"Recomputed roofline_pclass agrees with F-DATA's own precomputed pclass "
      f"in {agreement:.4%} of jobs")
print(f"Ridge point (single-node): "
      f"{roofline.A64FX_PEAK_DP_FLOPS_PER_NODE / roofline.A64FX_PEAK_HBM_BW_PER_NODE:.2f} FLOP/byte")

### Roofline chart

Both axes are per-node quantities: `opint` is already node-count invariant (both `flops` and `mbwidth` scale by `nnuma` equally, so the ratio doesn't), and achieved performance is divided by `nnuma` so every job — single-node or 150,000-node — plots against the same fixed ceiling curve.

In [ ]:
fig = plotting.plot_roofline(
    fdata_roofline["opint"].to_numpy(),
    fdata_roofline["achieved_flops_per_sec_per_node"].to_numpy(),
    roofline.A64FX_PEAK_DP_FLOPS_PER_NODE,
    roofline.A64FX_PEAK_HBM_BW_PER_NODE,
    pclass=fdata_roofline["pclass"].to_numpy(),
)
plt.show()

## Roofline as an execution-time baseline

The ceiling implies a best-case duration for each job: `flops / roofline_ceiling_flops_per_sec` — how long the job would take if it ran at the fastest rate its own measured operational intensity allows. Evaluated on the test split, against the naive per-user-median baseline (Decision #17) computed from the training split only.

In [ ]:
fdata_test_roofline = roofline.hierarchical_roofline_fdata(fdata_test)
valid = np.isfinite(fdata_test_roofline["roofline_predicted_duration"].to_numpy())
print(f"Valid Roofline predictions: {valid.sum():,} / {len(fdata_test_roofline):,}")

roofline_metrics = metrics.regression_metrics(
    fdata_test_roofline.loc[valid, "duration"].to_numpy(),
    fdata_test_roofline.loc[valid, "roofline_predicted_duration"].to_numpy(),
)
print("\nRoofline baseline (F-DATA execution time, test period):")
for k, v in roofline_metrics.items():
    print(f"  {k}: {v:,.4f}")

user_med, glob_med = baselines.fit_naive_baseline(fdata_train, "fdata", "duration")
naive_pred = baselines.predict_naive_baseline(fdata_test, "fdata", user_med, glob_med)
naive_metrics = metrics.regression_metrics(fdata_test["duration"].to_numpy(), naive_pred)
print("\nNaive per-user-median baseline (F-DATA execution time, test period):")
for k, v in naive_metrics.items():
    print(f"  {k}: {v:,.4f}")

# Part 2: PM100 — Calibrated Resource-Utilization Power Model (Decision #15)

PM100's schema has zero FLOP/instruction/performance-counter fields, so no Roofline-family metric is computable here — confirmed against `documentation/job_features.md`, not assumed. Instead: a resource-utilization power model whose functional form is fixed by hardware reasoning (an idle baseline plus a linear contribution per allocated resource type), with only the coefficients calibrated against measured power on a training-period-only subset.

**Correction to a prior assumption:** `node_power_consumption`'s name — and an earlier pass over this codebase — suggested it's a genuine per-node reading. It isn't. Checked directly below: dividing each job's mean power by `num_nodes_alloc` gives a stable per-node figure across every node count from 1 to 32+, which is only possible if the recorded value is already summed across all allocated nodes at each 20-second sample — the same kind of job-total-not-per-node quantity `avgpcon` turned out to be in F-DATA (notebook 02). The calibrated model below is built for the corrected, job-total interpretation: `P_idle` scales by `num_nodes_alloc` rather than being applied once per job.

In [ ]:
for n in [1, 2, 4, 8, 16, 32]:
    sub = pm100[pm100["num_nodes_alloc"] == n]
    if len(sub):
        total = sub["power_mean"].mean()
        print(f"{n:>3} nodes: mean total power {total:>10.1f} W "
              f"-> per-node {total/n:>7.1f} W  (n={len(sub):,})")

### Figure: per-node-normalized power holds roughly constant across node counts

If `node_power_consumption` were already per-node, this bar chart would show power falling or rising with node count as jobs at different scales have different characteristics. Instead it should stay roughly flat around Marconi100's known per-node draw — evidence the raw field is a job-wide total.

In [ ]:
node_counts = [1, 2, 4, 8, 16, 32]
per_node_means = []
labels = []
for n in node_counts:
    sub = pm100[pm100["num_nodes_alloc"] == n]
    if len(sub):
        per_node_means.append(sub["power_mean"].mean() / n)
        labels.append(str(n))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(labels, per_node_means)
ax.set_xlabel("num_nodes_alloc")
ax.set_ylabel("mean power_mean / num_nodes_alloc  (W)")
ax.set_title("PM100 node_power_consumption is a job-wide total, not per-node")
fig.tight_layout()
plt.show()

### A caveat before calibrating: the four predictors are highly correlated

Marconi100 nodes have a fixed core:GPU:memory ratio, so jobs requesting more of one resource almost always request proportionally more of the others. This matters for how the calibrated coefficients should be read — an OLS fit can land on odd-looking (even negative) individual coefficients under near-collinear inputs even when the model's overall predictions are reasonable, because the fit can't cleanly attribute the outcome to one input over its correlated neighbors.

In [ ]:
predictor_cols = ["num_nodes_alloc", "num_cores_alloc", "num_gpus_alloc", "mem_alloc"]
print(pm100_train[predictor_cols].corr())

## Calibration

`calibrate_pm100_power_model` fits `P_idle, alpha, beta, gamma` via OLS on `pm100_train` only — the test split is never touched by this step.

In [ ]:
coeffs = roofline.calibrate_pm100_power_model(pm100_train)
print(coeffs)
print(f"\nPlausibility check — published TDPs: 2x POWER9 CPU ~= 380W, 4x V100 GPU ~= 1200W per node.")
print(f"Calibrated P_idle: {coeffs.p_idle:.1f} W")
print(f"Calibrated alpha (per core): {coeffs.alpha:.3f} W, beta (per GPU): {coeffs.beta:.3f} W, "
      f"gamma (per MB mem): {coeffs.gamma:.5f} W")

### Figure: predicted vs. actual power on the calibration (training-period) subset

PM100's equivalent of a Roofline chart — this is the evidence that the analytical prior is sound, in place of a Roofline plot (which PM100 can't support).

In [ ]:
pred_train = roofline.predict_pm100_power(pm100_train, coeffs)
fig = plotting.plot_predicted_vs_actual(
    pm100_train["power_mean"].to_numpy(), pred_train,
    "PM100 calibrated power model — training-period calibration subset",
)
plt.show()

train_metrics = metrics.regression_metrics(pm100_train["power_mean"].to_numpy(), pred_train)
print("Calibration-subset fit:")
for k, v in train_metrics.items():
    print(f"  {k}: {v:,.4f}")

## Evaluation on the held-out test split, against the naive baseline

In [ ]:
pred_test = roofline.predict_pm100_power(pm100_test, coeffs)
power_model_metrics = metrics.regression_metrics(pm100_test["power_mean"].to_numpy(), pred_test)
print("Calibrated power model (PM100 power, test period):")
for k, v in power_model_metrics.items():
    print(f"  {k}: {v:,.4f}")

pm100_user_med, pm100_glob_med = baselines.fit_naive_baseline(pm100_train, "pm100", "power_mean")
pm100_naive_pred = baselines.predict_naive_baseline(pm100_test, "pm100", pm100_user_med, pm100_glob_med)
pm100_naive_metrics = metrics.regression_metrics(pm100_test["power_mean"].to_numpy(), pm100_naive_pred)
print("\nNaive per-user-median baseline (PM100 power, test period):")
for k, v in pm100_naive_metrics.items():
    print(f"  {k}: {v:,.4f}")

# Summary

**F-DATA's Roofline and PM100's power model are not directly comparable** — different formulas, different physical quantities, different targets (F-DATA: execution time; PM100: power). The table below places them side by side purely for this notebook's own recordkeeping, not as a cross-dataset claim.

In [ ]:
summary = pd.DataFrame([
    {"dataset": "F-DATA", "target": "duration (s)", "model": "Roofline", **roofline_metrics},
    {"dataset": "F-DATA", "target": "duration (s)", "model": "naive (user median)", **naive_metrics},
    {"dataset": "PM100", "target": "power_mean (W)", "model": "calibrated power model", **power_model_metrics},
    {"dataset": "PM100", "target": "power_mean (W)", "model": "naive (user median)", **pm100_naive_metrics},
])
summary

## Sanity-check assertions (Decision #19)

Cheap, targeted checks against the most damaging silent-failure classes in this notebook's custom code — not a full test suite.

In [ ]:
# Arithmetic intensity must be non-negative and finite for every job.
assert (fdata_roofline["opint"] >= 0).all(), "negative opint found"
assert np.isfinite(fdata_roofline["opint"].to_numpy()).all(), "non-finite opint found"

# Roofline ceiling can never exceed the compute-bound (peak-FLOPs) ceiling.
peak_flops_col = fdata_roofline["nnuma"] * roofline.A64FX_PEAK_DP_FLOPS_PER_NODE
assert (fdata_roofline["roofline_ceiling_flops_per_sec"] <= peak_flops_col + 1e-6).all(), (
    "roofline ceiling exceeds the compute-bound peak"
)

# Roofline-predicted duration must be positive wherever it's defined.
finite_pred = fdata_roofline["roofline_predicted_duration"].replace([np.inf, -np.inf], np.nan).dropna()
assert (finite_pred > 0).all(), "non-positive roofline_predicted_duration found"

# PM100 calibrated coefficients must be finite (a degenerate fit would silently
# produce NaN/inf and every downstream prediction would be garbage).
for name, value in [("p_idle", coeffs.p_idle), ("alpha", coeffs.alpha),
                    ("beta", coeffs.beta), ("gamma", coeffs.gamma)]:
    assert np.isfinite(value), f"PM100 power model coefficient {name} is not finite: {value}"

# log1p/expm1 round-trip (Decision #3/#19), reusing the actual targets involved here.
metrics.expm1_round_trip_check(fdata_test["duration"].to_numpy())
metrics.expm1_round_trip_check(pm100_test["power_mean"].to_numpy())

# Chronological split must not leak: every train row's time must be <= every test row's time boundary.
fdata_boundary = splits.split_boundary(fdata, "fdata")
assert (pd.to_datetime(fdata_train["adt"]) <= fdata_boundary).all()
assert (pd.to_datetime(fdata_test["adt"]) > fdata_boundary).all()
pm100_boundary = splits.split_boundary(pm100, "pm100")
assert (pm100_train["submit_time"] <= pm100_boundary).all()
assert (pm100_test["submit_time"] > pm100_boundary).all()

print("All sanity checks passed.")

## Deferred to later notebooks

- Full-scale F-DATA Roofline (all 38 months) — this notebook uses a 6-month dev-scale slice, per Decision #10.
- The rolling-window temporal-drift check (Decision #18) across F-DATA's full span — a separate check from the single chronological split used here.
- Both baselines (Roofline, naive median, calibrated power model) get reused in notebook 08's evaluation sweep, where they sit alongside RF/XGBoost/LightGBM/FNN/LSTM/TCN/Hybrid in the same comparison tables.